In [1]:
from oqd_compiler_infrastructure import Post, PrettyPrint
from oqd_core.analysis.analog.cfg import AnalogCFGBuilder
from oqd_core.analysis.analog.symbol_table import AnalogSymbolTableBuilder
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.compiler.analog.passes.compile import compile_analog_circuit
from oqd_core.frontend.analog import parse_analog

from oqd_analog_emulator.interpreter import AnalogInterpreter

printer = Post(PrettyPrint())

source = """ 
r = qreg(5)
q0 = r[0]
q1 = r[1]
q2 = r[2]
q3 = r[3]
initialize(r)
// s = [q0, q1]
s = [q1, q0]
u = [q2, q3]
t = [q0, q2]
// result = evolve(%X , 1, q0)
result = evolve( %X %@ %X, 1, s)
result2 = evolve( %X %@ %X, 1, u)
result3 = evolve( %X %@ %X, 1, t)
// evolve(%X, 1, r[2])
"""


circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
checker = AnalogTypeChecker(cfg)

symbol_analysis = AnalogSymbolTableBuilder(cfg, checker.dataflow_result)
symbol_table = symbol_analysis.symbol_table

circuit, cfg = compile_analog_circuit(
    circuit=circuit, cfg=cfg, symbol_table=symbol_table
)


In [2]:
interpreter = AnalogInterpreter(method_table="QutipMethodTable", fock_cutoff=4, dt=1e-4)
interpreter.run(cfg=cfg)

[]

In [3]:
store = interpreter.get_store()
store
# interpreter.vm.stack
# store['q0'].state

{'r': [<ListTerminators.LISTSTART: 0>,
  RegisterName(name='r', index=0, dim=2),
  RegisterName(name='r', index=1, dim=2),
  RegisterName(name='r', index=2, dim=2),
  RegisterName(name='r', index=3, dim=2),
  RegisterName(name='r', index=4, dim=2),
  <ListTerminators.LISTEND: 1>],
 'q0': RegisterName(name='r', index=0, dim=2),
 'q1': RegisterName(name='r', index=1, dim=2),
 'q2': RegisterName(name='r', index=2, dim=2),
 'q3': RegisterName(name='r', index=3, dim=2),
 's': [<ListTerminators.LISTSTART: 0>,
  RegisterName(name='r', index=1, dim=2),
  RegisterName(name='r', index=0, dim=2),
  <ListTerminators.LISTEND: 1>],
 'u': [<ListTerminators.LISTSTART: 0>,
  RegisterName(name='r', index=2, dim=2),
  RegisterName(name='r', index=3, dim=2),
  <ListTerminators.LISTEND: 1>],
 't': [<ListTerminators.LISTSTART: 0>,
  RegisterName(name='r', index=0, dim=2),
  RegisterName(name='r', index=2, dim=2),
  <ListTerminators.LISTEND: 1>],
 'result': [<ListTerminators.LISTSTART: 0>, <ListTerminators.L

In [4]:
# from oqd_core.compiler.analog.math.rules import SubstituteMathVar
# from oqd_core.interface.analog.expr import MathVar
# from oqd_compiler_infrastructure import Post
# substitute_pass = Post(SubstituteMathVar(MathVar(class_='MathVar', name='#s'), MathVar(class_='MathVar', name='#t') - 10))

# substitute_pass(store['a'])
registers = interpreter.vm.registers
registers

{RegisterName(name='r', index=0, dim=2): QuantumRegister(name=[RegisterName(name='r', index=0, dim=2), RegisterName(name='r', index=1, dim=2), RegisterName(name='r', index=2, dim=2), RegisterName(name='r', index=3, dim=2)], time=3.0, time_last_updated=3.0, state=Quantum object: dims=[[2, 2, 2, 2], [1]], shape=(16, 1), type='ket', dtype=Dense
 Qobj data =
 [[ 0.15772854+0.j        ]
  [ 0.        +0.j        ]
  [ 0.        +0.j        ]
  [ 0.        -0.2456478j ]
  [ 0.        +0.j        ]
  [ 0.        +0.59582334j]
  [-0.38257352+0.j        ]
  [ 0.        +0.j        ]
  [ 0.        +0.j        ]
  [-0.38257404+0.j        ]
  [ 0.        -0.24564781j]
  [ 0.        +0.j        ]
  [ 0.        -0.24564747j]
  [ 0.        +0.j        ]
  [ 0.        +0.j        ]
  [-0.38257352+0.j        ]]),
 RegisterName(name='r', index=1, dim=2): QuantumRegister(name=[RegisterName(name='r', index=0, dim=2), RegisterName(name='r', index=1, dim=2), RegisterName(name='r', index=2, dim=2), RegisterN

In [5]:
cfg.to_dict()

{0: {'register_id': 0,
  'kind': 'start',
  'stmt': {},
  'preds': [],
  'succs': [1],
  'exit_nodes': [],
  'edge_labels': {}},
 1: {'register_id': 1,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'r',
   'value': {'class_': 'QuantumRegister', 'size': 5}},
  'preds': [0],
  'succs': [2],
  'exit_nodes': [],
  'edge_labels': {}},
 2: {'register_id': 2,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q0',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 0}},
  'preds': [1],
  'succs': [3],
  'exit_nodes': [],
  'edge_labels': {}},
 3: {'register_id': 3,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q1',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 1}},
  'preds': [2],
  'succs': [4],
  'exit_nodes': [],
  'edge_labels': {}},
 4: {'register_id': 4,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q2',
   'value': {'cla